In [ ]:
# Optional local install if you want to rerun API generation:
# pip install together openai


## Reproduction Notes
This notebook handles the WorldValue synthetic-response generation and post-processing pipeline used in the paper.

- The default setting is artifact-first: `RUN_GENERATION = False`, so the notebook can be opened and stepped through without making provider API calls.
- If you want to regenerate model outputs, set `RUN_GENERATION = True`, provide credentials in `.env.local`, and supply your own generation backend for `run_generate_responses_async_unified(...)`.
- The committed reproduction subset is `data/worldvalue/retained_questions_235.json`, which matches the final 235-question experiment used downstream.


In [ ]:
import asyncio
import gc
import glob
import json
import os
import pickle
import random
import re
import sys
import time
from collections import OrderedDict, defaultdict, deque
from difflib import SequenceMatcher
from pathlib import Path
from typing import Dict, List, Optional

NOTEBOOK_DIR = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    direct = candidate / 'wvs_notebook_helpers.py'
    nested = candidate / 'paper_reproduction' / 'worldvalue_quantile' / 'wvs_notebook_helpers.py'
    if direct.exists():
        NOTEBOOK_DIR = candidate
        break
    if nested.exists():
        NOTEBOOK_DIR = nested.parent
        break
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate paper_reproduction/worldvalue_quantile.')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np
import pandas as pd

from wvs_notebook_helpers import filter_mapping_to_questions, install_numpy_pickle_compat, load_retained_questions

run_generate_responses_async_unified = None

install_numpy_pickle_compat()


### Repository Root Setup
The notebook locates the repository root automatically so it can be run from this folder or from the repo root.


In [ ]:
import os, sys
from pathlib import Path

NOTEBOOK_DIR = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    direct = candidate / 'wvs_notebook_helpers.py'
    nested = candidate / 'paper_reproduction' / 'worldvalue_quantile' / 'wvs_notebook_helpers.py'
    if direct.exists():
        NOTEBOOK_DIR = candidate
        break
    if nested.exists():
        NOTEBOOK_DIR = nested.parent
        break
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate paper_reproduction/worldvalue_quantile.')

from wvs_notebook_helpers import find_repo_root

ROOT = find_repo_root(NOTEBOOK_DIR)
os.chdir(ROOT)
for entry in [NOTEBOOK_DIR, ROOT]:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))
print('Using reproduction root:', ROOT)
print('Using notebook directory:', NOTEBOOK_DIR)


In [ ]:
def _load_dotenv_local(path: str = ".env.local") -> None:
    import os
    from pathlib import Path

    base = Path.cwd().resolve()
    candidates = []
    for p in [base, *base.parents]:
        candidates.append(p / path)
        candidates.append(p / ".env.local.example")

    seen = set()
    for fp in candidates:
        if fp in seen or not fp.exists():
            continue
        seen.add(fp)
        for raw in fp.read_text(encoding="utf-8").splitlines():
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[len("export ") :].strip()
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            k = k.strip()
            v = v.strip().strip("\"").strip("'")
            if k and v and not os.getenv(k):
                os.environ[k] = v

def resolve_api_key(api_platform: str, api_key: str | None = None) -> str:
    """
    Return a non-empty API key for the chosen platform.
    - If api_key is truthy, use it.
    - Else fall back to env: OPENAI_API_KEY / TOGETHER_API_KEY / GOOGLE_API_KEY.
    """
    import os

    _load_dotenv_local(".env.local")

    plat = (api_platform or "").strip().lower()
    if api_key:
        return api_key

    if plat in ("openai", "oai"):
        k = os.getenv("OPENAI_API_KEY", "")
    elif plat in ("together", "togetherai"):
        k = os.getenv("TOGETHER_API_KEY", "")
    elif plat in ("gemini", "google", "googleai", "google-ai"):
        k = os.getenv("GOOGLE_API_KEY", "") or os.getenv("GEMINI_API_KEY", "")
    else:
        k = ""

    if not k:
        raise ValueError(
            f"Missing API key for platform '{api_platform}'. "
            f"Provide `api_key=` or set the appropriate environment variable."
        )
    return k


def mask_key(k: str) -> str:
    if not k:
        return "<EMPTY>"
    if len(k) <= 6:
        return "*" * len(k)
    return k[:3] + "*" * (len(k) - 6) + k[-3:]


# ========= add this: a tiny fatal-error classifier + abort exception =========
class RunAbort(Exception):
    """Raise this to abort the entire simulation early (e.g., 401, insufficient_quota)."""

    def __init__(self, reason: str):
        super().__init__(reason)
        self.reason = reason


def _is_fatal_error(e: Exception, provider: str) -> str | None:
    """
    Returns a short reason string if the error is fatal; otherwise None.
    We match on common messages/codes from OpenAI/Together SDKs.
    """
    msg = str(e)
    low = msg.lower()

    # Auth / key missing or invalid
    if "authenticationerror" in low or "no api key" in low or "you didn't provide an api key" in low:
        return "Authentication error (no/invalid API key)."

    if "401" in low and ("auth" in low or "unauthorized" in low):
        return "Unauthorized (401)."

    # Quota/billing
    if "insufficient_quota" in low or "exceeded your current quota" in low:
        return "Insufficient quota/billing."

    if "billing" in low and "quota" in low and "exceeded" in low:
        return "Insufficient quota/billing."

    # Model issues
    if "model not found" in low or "does not exist" in low:
        return "Model not found / unavailable."

    if provider == "together" and ("access denied" in low or "forbidden" in low or "payment" in low):
        return "Together access denied / billing issue."

    return None



In [ ]:
# Load from Colab userdata if available; otherwise fall back to local .env.local
try:
    openai_key = userdata.get('OPENAI_API_KEY')
    together_key = userdata.get('TOGETHER_API_KEY')
    gemini_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    openai_key = together_key = gemini_key = None

if not any([openai_key, together_key, gemini_key]):
    _load_dotenv_local('.env.local')

# Set environment variables for SDKs
if openai_key:
    os.environ['OPENAI_API_KEY'] = openai_key
if together_key:
    os.environ['TOGETHER_API_KEY'] = together_key
if gemini_key:
    os.environ['GOOGLE_API_KEY'] = gemini_key

if os.environ.get('GOOGLE_API_KEY') and not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = os.environ['GOOGLE_API_KEY']

print('OpenAI key loaded:', bool(os.environ.get('OPENAI_API_KEY')))
print('Together key loaded:', bool(os.environ.get('TOGETHER_API_KEY')))
print('Gemini key loaded:', bool(os.environ.get('GOOGLE_API_KEY')))



In [ ]:
# Default mode: do not hit provider APIs unless you opt in.
RUN_GENERATION = False

# Qwen3 235B A22B Instruct 2507 FP8
api_platform = 'togetherai'
api_key = ''
model_name = 'Qwen/Qwen3-235B-A22B-Instruct-2507-tput'
model_name_short = 'Qwen3-235B'

'''
api_platform = 'togetherai'
api_key = ''
model_name = "meta-llama/Llama-3.3-70B-Instruct-Turbo-Free"
model_name_short = "llama-3.3"
'''
num_of_synthetic_answers = 500  # raw generation budget per question before analysis-time subsampling

# specify the indices of questions
num_groups = 10  # partition all questions into this number of groups
id_group = 1     # work on this group of questions

# paths for saving raw and cleaned results
file_path_raw = 'data/worldvalue/synthetic answers/raw/synthetic_answers_raw_{}-{}.pkl'.format(model_name_short, id_group)
file_path_clean = 'data/worldvalue/synthetic answers/clean/synthetic_answers_clean_{}-{}.pkl'.format(model_name_short, id_group)


In [ ]:
# load retained question ids and committed artifacts
questions = load_retained_questions()
with open('data/worldvalue/synthetic_profiles.pkl', 'rb') as f:
    synthetic_profiles = filter_mapping_to_questions(pickle.load(f), questions)
with open('data/worldvalue/choices_to_numeric.json', 'r') as f:
    choices_to_numeric = filter_mapping_to_questions(json.load(f), questions)

# specify the indices of questions
m = len(questions)
tmp = m // num_groups
if id_group < num_groups:
    idx_questions = np.arange((id_group - 1) * tmp, id_group * tmp)
else:
    idx_questions = np.arange((id_group - 1) * tmp, m)
print(f'Retained questions: {len(questions)}')
print('This group: {} questions.'.format(len(idx_questions)))


In [ ]:
id_groups = [7, 8, 9, 10]

if not RUN_GENERATION:
    print('Skipping provider API generation because RUN_GENERATION=False. Set it to True if you want to regenerate raw model outputs.')
elif run_generate_responses_async_unified is None:
    raise RuntimeError('run_generate_responses_async_unified(...) is not bundled in this reproduction folder. Set RUN_GENERATION=False to use the committed artifacts, or provide your own generation backend before rerunning this cell.')
else:
    for id_group in id_groups:
        m = len(questions)
        tmp = m // num_groups
        if id_group < num_groups:
            idx_questions = np.arange((id_group - 1) * tmp, id_group * tmp)
        else:
            idx_questions = np.arange((id_group - 1) * tmp, m)

        print(f'This group: {len(idx_questions)} questions (group {id_group}/{num_groups}).')

        file_path_raw = 'data/worldvalue/synthetic answers/raw/synthetic_answers_raw_{}-{}.pkl'.format(model_name_short, id_group)
        API_KEY = resolve_api_key(api_platform, api_key)
        print('Using API key:', mask_key(API_KEY))

        results = run_generate_responses_async_unified(
            api_platform=api_platform,
            api_key=API_KEY,
            model_name=model_name,
            questions=questions,
            idx_questions=idx_questions,
            synthetic_profiles=synthetic_profiles,
            num_of_synthetic_answers=num_of_synthetic_answers,
            file_path=file_path_raw,
            instruction='You are simulating the behaviors of humans with certain specified characteristics to help with a survey study.',
            concurrency=4,
            qps=5,
            rpm=150,
            batch_size=25,
        )


In [ ]:
# --- Convert to Batched Synthetic Files to One ---
def to_pystr(obj):
    """Recursively coerce numpy string types to plain Python str."""
    if isinstance(obj, dict):
        return {str(k): to_pystr(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_pystr(x) for x in obj]
    if isinstance(obj, tuple):
        return tuple(to_pystr(x) for x in obj)
    # cast numpy scalar strings to str
    if isinstance(obj, (np.str_, np.bytes_)):
        return str(obj)
    return obj

def load_pkl(p):
    with open(p, "rb") as f:
        obj = pickle.load(f)
    return to_pystr(obj)

def save_pkl(obj, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    pickle.dump(to_pystr(obj), open(p, "wb"))

def concat_synth_answer_shards_preserve_keys(
    pattern: str,          # e.g., "/content/.../synthetic_answers_raw_llama-3.3-*.pkl"
    out_path: str,         # e.g., "/content/.../synthetic_answers_raw_llama-3.3-all.pkl"
    dedupe: bool = False
):
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files match: {pattern}")

    combined: dict[str, list] = OrderedDict()

    for fp in files:
        shard = load_pkl(fp)  # already normalized to str keys/values where needed
        if not isinstance(shard, dict):
            print(f"[warn] Skipping {fp}: not a dict.")
            continue

        for qid, lst in shard.items():
            qid_key = str(qid)  # ensure plain str key
            if qid_key not in combined:
                combined[qid_key] = []

            if isinstance(lst, list):
                if not dedupe:
                    combined[qid_key].extend(lst)
                else:
                    seen = set(combined[qid_key])
                    for s in lst:
                        s_norm = to_pystr(s)
                        if s_norm not in seen:
                            combined[qid_key].append(s_norm)
                            seen.add(s_norm)
            else:
                print(f"[warn] {fp}:{qid_key} value is not a list, skipping.")

    save_pkl(dict(combined), out_path)
    print(f"[done] merged {len(files)} files → {out_path}")
    print(f"QIDs: {len(combined)} | total responses: {sum(len(v) for v in combined.values())}")


def _sim(a, b):  # fuzzy similarity
    return SequenceMatcher(None, str(a), str(b)).ratio()

def _is_number_like(x):
    try:
        float(x)
        return True
    except Exception:
        return False

def _upgrade_to_value_map(label_map: dict):
    """
    Turn a per-QID label map into a value map usable for direct lookup:
      - If values are already numeric (floats/ints in [-1,1]), keep them.
      - Else if values look like integer codes, linearly scale codes to [-1,1]
        with low code -> +1, high code -> -1 (WVS-style).
      - Also add numeric-string keys: "1","2",... -> value, so [[N]] works.
    """
    if not label_map:
        return {}

    vals = list(label_map.values())
    # Case A: already numeric values (floats)
    if all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals) and \
       all(-1.0000001 <= float(v) <= 1.0000001 for v in vals):
        value_map = {str(k): float(v) for k, v in label_map.items()}
        return value_map

    # Case B: treat as codes; scale to [-1,1]
    # Collect codes that are number-like from the map values
    codes = []
    for v in vals:
        if _is_number_like(v):
            codes.append(int(float(v)))
    if not codes:
        # nothing we can do—return empty map (caller will output NaN)
        return {}

    low, high = min(codes), max(codes)
    step = max(1, high - low)

    def scale(code_int):
        # low -> +1, high -> -1
        t = (code_int - low) / step
        return float(1.0 - 2.0 * t)

    # Build: label -> scaled_value
    value_map = {}
    for label, v in label_map.items():
        if _is_number_like(v):
            code = int(float(v))
            value_map[str(label)] = scale(code)

    # Also: numeric strings "1","2",... -> scaled_value
    for code in range(low, high + 1):
        value_map[str(code)] = scale(code)

    # Handle common “refused / don’t know” if present in labels
    for label in label_map:
        lbl = str(label).strip().lower()
        if any(k in lbl for k in ["refused", "don't know", "dont know", "dk", "na", "no answer"]):
            value_map[str(label)] = 0.0

    return value_map

def process_responses_WorldValue(
    synthetic_answers_raw,     # dict: {qid: [raw_strings]}
    questions,                 # list of qids (e.g., ["Q114", ...])
    idx_questions,             # list of indices into `questions`
    choices_to_numeric,        # dict: {qid: {choice_text -> (value OR code)}}
    file_path                  # pickle path for numeric outputs
):
    """
    Converts raw answers to numeric values using ONLY `choices_to_numeric` per QID.
    - If choices_to_numeric[qid] gives label->final_value: use directly.
    - If it gives label->code: auto-scale codes (low -> +1, high -> -1) per QID.
    - Supports numeric answers ([[N]] or bare numbers) via added "N" string keys.
    """
    # resume partial results
    synthetic_answers_numeric = {}
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            synthetic_answers_numeric = pickle.load(f)

    RE_DBL = re.compile(r"\[\[\s*([0-9]+)\s*")   # [[ 3 ]]
    RE_ANY = re.compile(r"(?<!\d)(\d{1,2})(?!\d)")

    for i in idx_questions:
        qid = questions[i]

        if qid in synthetic_answers_numeric:
            print(f"Question {i} finished before.")
            continue

        print(f"Working on Question {i} ({qid}).")
        raw_list = synthetic_answers_raw.get(qid, [])
        if not raw_list:
            print(f"  Warning: no raw answers for {qid}; skipping.")
            synthetic_answers_numeric[qid] = []
            with open(file_path, 'wb') as f:
                pickle.dump(synthetic_answers_numeric, f)
            continue

        # Build a per-QID value map (labels and numeric-string keys -> final value)
        label_map = choices_to_numeric.get(qid, {}) or {}
        value_map = _upgrade_to_value_map(label_map)

        # Separate label keys for fuzzy fallback
        label_keys = [k for k in value_map.keys() if not _is_number_like(k)]

        numeric_vals = []
        for resp in raw_list:
            s = str(resp or "")
            val = np.nan

            # 1) Prefer [[N]]
            m = RE_DBL.search(s)
            if m:
                code_str = m.group(1)
                val = value_map.get(code_str, np.nan)
            else:
                # 2) Any bare number
                m2 = RE_ANY.search(s)
                if m2:
                    code_str = m2.group(1)
                    val = value_map.get(code_str, np.nan)

            # 3) Fuzzy to labels if still NaN
            if np.isnan(val) and label_keys:
                sims = [_sim(s, lbl) for lbl in label_keys]
                best = int(np.argmax(sims))
                val = value_map.get(label_keys[best], np.nan)

            numeric_vals.append(val)

        synthetic_answers_numeric[qid] = numeric_vals
        with open(file_path, 'wb') as f:
            pickle.dump(synthetic_answers_numeric, f)
        print(f"  Saved {qid}: {len(numeric_vals)} answers.")

    print("Post-processing completed.")
    return synthetic_answers_numeric




In [ ]:
# ---- merge raw shard files if they exist ----
pattern = 'data/worldvalue/synthetic answers/raw/synthetic_answers_raw_{}-*.pkl'.format(model_name_short)
out_path = 'data/worldvalue/synthetic answers/raw/synthetic_answers_raw_{}.pkl'.format(model_name_short)
matching_files = sorted(glob.glob(pattern))

if not matching_files:
    print(f'No shard files matched {pattern}. Skipping merge because no raw shard files were found for this model.')
else:
    concat_synth_answer_shards_preserve_keys(pattern, out_path, dedupe=False)


In [ ]:
# Next step: Convert them to numerics
# --- small helpers ---
_RE_DBL = re.compile(r"\[\[\s*([0-9]{1,2})\s*")          # [[ 3 ]]
_RE_ANY = re.compile(r"(?<!\d)(\d{1,2})(?!\d)")          # standalone 1–2 digit
_WORD2NUM = {
    "zero":"0","none":"0",
    "one":"1","two":"2","three":"3","four":"4","five":"5",
    "six":"6","seven":"7","eight":"8","nine":"9","ten":"10",
}

def _extract_code_str(text: str, valid_codes: set[str]) -> str | None:
    """Return the first code as a string that exists in valid_codes, else None."""
    s = str(text or "")
    # 1) [[N]]
    m = _RE_DBL.search(s)
    if m:
        c = m.group(1)
        if c in valid_codes:
            return c
    # 2) bare number
    m2 = _RE_ANY.search(s)
    if m2:
        c = m2.group(1)
        if c in valid_codes:
            return c
    # 3) spelled number
    s_lower = " " + s.lower() + " "
    for w, c in _WORD2NUM.items():
        if f" {w} " in s_lower and c in valid_codes:
            return c
    return None

# --- main transformer (no num_to_num needed) ---
def process_responses_WorldValue_numeric_only(
    synthetic_answers_raw,         # dict OR pickle path: {qid: [raw_strings]}
    questions,                     # list of qids (e.g., ["Q1","Q2",...])
    idx_questions,                 # indices into `questions`
    choices_to_numeric,            # dict: {qid: {'1':1.0,'2':0.3333,...}}
    out_path                       # pickle path to write numeric results
):
    # load raw dict if a path was passed
    if isinstance(synthetic_answers_raw, str):
        with open(synthetic_answers_raw, "rb") as f:
            synthetic_answers_raw = pickle.load(f)

    # resume numeric dict if present
    numeric_out = {}
    if os.path.exists(out_path):
        with open(out_path, "rb") as f:
            numeric_out = pickle.load(f)

    for i in idx_questions:
        qid = questions[i]
        if qid in numeric_out:
            print(f"Question {i} ({qid}) finished before.")
            continue

        print(f"Working on Question {i} ({qid}).")
        raw_list = synthetic_answers_raw.get(qid, [])
        if not raw_list:
            print(f"  Warning: no raw answers for {qid}; skipping.")
            numeric_out[qid] = []
            with open(out_path, "wb") as f:
                pickle.dump(numeric_out, f)
            continue

        # per-QID mapping: code(str) -> final numeric
        mp = choices_to_numeric.get(qid, {}) or {}
        valid_codes = set(str(k) for k in mp.keys())

        vals = []
        for resp in raw_list:
            code = _extract_code_str(resp, valid_codes)
            vals.append(mp.get(code, np.nan))

        numeric_out[qid] = vals

        # checkpoint after each qid
        with open(out_path, "wb") as f:
            pickle.dump(numeric_out, f)
        print(f"  Saved {qid}: {len(vals)} answers.")

    print("Post-processing completed.")
    return numeric_out


In [ ]:
# paths
raw_path = 'data/worldvalue/synthetic answers/raw/synthetic_answers_raw_{}.pkl'.format(model_name_short)
clean_path = 'data/worldvalue/synthetic answers/clean/synthetic_answers_clean_{}.pkl'.format(model_name_short)

questions = load_retained_questions()
idx_questions = list(range(len(questions)))

if not Path(raw_path).exists():
    print(f'Raw merged file not found at {raw_path}. Skipping numeric conversion because the merged raw file is not present.')
else:
    numeric_dict = process_responses_WorldValue_numeric_only(
        synthetic_answers_raw=raw_path,
        questions=questions,
        idx_questions=idx_questions,
        choices_to_numeric=choices_to_numeric,
        out_path=clean_path,
    )
    numeric_dict = filter_mapping_to_questions(numeric_dict, questions)
    with open(clean_path, 'wb') as f:
        pickle.dump(numeric_dict, f)
    print(f'Wrote cleaned retained-question synthetic-answer artifact to {clean_path}. Questions: {len(numeric_dict)}')
